# Agents Smoke Test

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parents[1]))

In [ ]:
import sys
from pprint import pprint

if ".." not in sys.path:
    sys.path.insert(0, "..")

### 1. Synthesize an alert

In [ ]:
from tools.bigquery_tools import get_alert_data

alert = get_alert_data("ALERT-001")
pprint(alert)

### 2. Build initial state

In [ ]:
from graph.state import ComplianceState

state: ComplianceState = {
    "alert": alert,
    "investigador_status": "pending",
}
pprint(state)

### 3. Run ResearchAgent

In [ ]:
from agents.research import ResearchAgent

agent = ResearchAgent()
result = agent.run(state)

print("investigador_status:", result["investigador_status"])
print("investigador_error: ", result["investigador_error"])

### 4. Inspect results

In [ ]:
print("customer:")
pprint(result["customer"])

print("\ntransaction_summary:")
pprint(result["transaction_summary"])

first_doc_id = result["documents"][0]["document_id"] if result["documents"] else "N/A"
print("\ndocuments retrieved:", len(result["documents"]), "| first document_id:", first_doc_id)

if result["documents"]:
    print("\nfirst document (first 300 chars):")
    print(result["documents"][0]["content"][:300])

### 5. Run RiskAnalyzerAgent

In [ ]:
from agents.risk_analyzer import RiskAnalyzerAgent

risk_agent = RiskAnalyzerAgent()
result = risk_agent.run(result)

print("risk_analyzer_status:", result["risk_analyzer_status"])
print("risk_analyzer_error: ", result["risk_analyzer_error"])


In [ ]:
print("risk_score:", result["risk_score"])
print("\nrisk_justification:")
print(result["risk_justification"])
print("\nanomalies:")
for a in result["anomalies"]:
    print(" -", a)
print("\nrisk_summary:")
print(result["risk_summary"])


### 6. Run DecisionAgent

In [ ]:
from agents.decision import DecisionAgent

decision_agent = DecisionAgent()
result = decision_agent.run(result)

print("decision_status:", result["decision_status"])
print("decision_error: ", result["decision_error"])

In [ ]:
print("decision:     ", result["decision"])
print("confidence:   ", result["confidence"])
print("\nreasoning_steps:")
for step in result["reasoning_steps"]:
    print(" -", step)
print("\nfinal_report:")
print(result["final_report"])
print("\napplicable_regulations:")
pprint(result["applicable_regulations"])